# 3 — BankNifty Options Charts (Auto-Refresh)

**Run frequency:** During market hours alongside Notebook 2

**What this does:**
1. Logs in to Zerodha KiteConnect (needed for live ATM strike)
2. Starts a SparkSession
3. Reads instruments + expiries from Parquet
4. Builds the options DataFrame for the configured ATM strike
5. Renders three Plotly charts:
   - **Price & ROC** — Straddle close + stop-loss + entry allowed + ROC
   - **Open Interest** — Combined, CE, PE OI
   - **Candlestick** — OHLC with date axis
6. **Auto-refreshes** every minute as new Parquet data arrives from Notebook 2

---
### ⚙️ Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `STRIKE_LEVEL_NAME` | `'ATM'` | Which strike to chart ('ATM', 'ATM+1', 'ATM-2', …) |
| `CE_OR_PE` | `'E'` | `'CE'`, `'PE'`, or `'E'` / `'S'` for straddle |
| `NUM_DAYS` | `2` | Days of data to display |
| `IS_LATEST_DAY` | `True` | Show only today's data |
| `CUSTOM_STRIKE` | `56500` | Override ATM (0 = live LTP) |
| `NUM_STRIKES` | `11` | Strikes each side of ATM to load |
| `LOOP_INTERVAL_MIN` | `1` | Chart refresh frequency |

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [2]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
STRIKE_LEVEL_NAME  = 'ATM'   # 'ATM', 'ATM+1', 'ATM-2', etc.
CE_OR_PE           = 'E'     # 'CE', 'PE', or 'E' for straddle
NUM_DAYS           = 2       # Days of OHLC data to show
IS_LATEST_DAY      = True    # True = show today only
NUM_DAYS_BACK      = 0       # 0 = latest, 1 = go back 1 day
CUSTOM_STRIKE      = 0       # 0 = use live LTP; override e.g. 56500
NUM_STRIKES        = 11      # Strikes each side of ATM to load
LOOP_INTERVAL_MIN  = 1       # Chart refresh frequency (minutes)
INTERVAL           = '3minute'

In [3]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

2026-03-11 00:44:54,513 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-11 00:44:54,682 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-11 00:44:54,829 [INFO] utils.kite_helpers — Login successful — request_token: A3XivZCbXhPLQeoknawOmxkKCI0iuQdA
2026-03-11 00:44:55,000 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [4]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='3_Charts_BNF')
print('✅ Spark session ready')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 00:45:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/11 00:45:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/11 00:45:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ Spark session ready


In [5]:
# ── Step 3: Load instruments and expiry data ───────────────────────────────
from utils.strike_utils import read_instruments, get_expiry_dates, get_Options_DF, get_ATM_Strike
from utils.strike_utils import BANKNIFTY_INDEX_TOKEN

bnf_options, bnf_futures, nifty_options, nifty_futures, expiries_df = read_instruments(spark)
bnf_expiries = get_expiry_dates(expiries_df, 'BANKNIFTY')

print(f"BankNifty current week expiry: {bnf_expiries['current_week']}")
print(f"BankNifty next week expiry:    {bnf_expiries['next_week']}")

BankNifty current week expiry: 2026-03-30
BankNifty next week expiry:    2026-04-28


In [6]:
# ── Step 4: Build options DataFrame for ATM region ────────────────────────
# Determine ATM strike (live or custom override)
if CUSTOM_STRIKE > 0:
    bnf_atm = CUSTOM_STRIKE
else:
    bnf_atm = get_ATM_Strike(kite, BANKNIFTY_INDEX_TOKEN, rounding_number=100)

print(f'BankNifty ATM Strike = {bnf_atm}')

Banknifty_Options_DF = get_Options_DF(
    spark         = spark,
    options_df_from_file = bnf_options,
    atm_strike    = bnf_atm,
    current_expiry= bnf_expiries['current_week'],
    strike_range  = 100,
    num_strikes   = NUM_STRIKES,
)
print(f"Options loaded: {Banknifty_Options_DF.count()} rows")
Banknifty_Options_DF.show(5)

BankNifty ATM Strike = 57000


26/03/11 00:45:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/03/11 00:45:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Old Generation, G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Options loaded: 46 rows


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py:154: DeprecationWarning: This process (pid=46260) is multi-threaded, use of fork() may lead to deadlocks in the child.


+---------+----------------+----------+-----------------+-------+-----------------+--------+---------------+-------+
|     name|instrument_token|    expiry|Strike_level_Name| strike|Strike_level_type|lot_size|instrument_type|segment|
+---------+----------------+----------+-----------------+-------+-----------------+--------+---------------+-------+
|BANKNIFTY|        13432834|2026-03-30|           ATM-11|55900.0|              ITM|      30|             CE|NFO-OPT|
|BANKNIFTY|        13433090|2026-03-30|           ATM-11|55900.0|              ITM|      30|             PE|NFO-OPT|
|BANKNIFTY|        13433346|2026-03-30|           ATM-10|56000.0|              ITM|      30|             CE|NFO-OPT|
|BANKNIFTY|        13434114|2026-03-30|           ATM-10|56000.0|              ITM|      30|             PE|NFO-OPT|
|BANKNIFTY|        13434370|2026-03-30|            ATM-9|56100.0|              ITM|      30|             CE|NFO-OPT|
+---------+----------------+----------+-----------------+-------

In [7]:
# ── Step 5: Start auto-refreshing charts ──────────────────────────────────
# This cell blocks. Press Kernel → Interrupt to stop.
from utils.chart_utils import run_chart_loop

run_chart_loop(
    spark                 = spark,
    options_df            = Banknifty_Options_DF,
    expiry                = bnf_expiries['current_week'],
    strike_level_name     = STRIKE_LEVEL_NAME,
    ce_or_pe              = CE_OR_PE,
    interval              = INTERVAL,
    num_days              = NUM_DAYS,
    is_latest_day         = IS_LATEST_DAY,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
)

26/03/11 00:45:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-03-11 00:45:41,621 [ERROR] utils.chart_utils — Chart refresh failed: 'DataFrame' object has no attribute 'CE_close_On_Open'
Traceback (most recent call last):
  File "/Users/yogeshpatil/Documents/Trading/CHEWY_BNF_Options_Trading/CHEWY_BNF_Options_Trading-1/PROD/utils/chart_utils.py", line 742, in _job
    get_chart(
  File "/Users/yogeshpatil/Documents/Trading/CHEWY_BNF_Options_Trading/CHEWY_BNF_Options_Trading-1/PROD/utils/chart_utils.py", line 701, in get_chart
    df_pd = compute_analytics(spark, raw_df, ce_or_pe, is_latest_day)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yogeshpatil/Documents/Trading/CHEWY_BNF_Options_Trading/CHEWY_BNF_Options_Trading-1/PROD/utils/chart_utils.py", line 453, in compute_analytics
    F.when(df.CE_close_On_Open >= df.PE

⚠️  Chart refresh error: 'DataFrame' object has no attribute 'CE_close_On_Open'


2026-03-11 00:46:14,958 [INFO] utils.chart_utils — Chart loop stopped by user.
